# 🏗️ OneVoice Edge — Synthetic Noisy Construction Speech Generator

**Pipeline:**
```
[8,064 Construction Utterances v2]
         ↓
   1. Multi-Speaker Neural TTS (Microsoft Edge TTS: Male/Female VI & EN)
         ↓
   [Clean Speech .wav]
         ↓
   2. Room Impulse Response (RIR / Reverb)
         ↓
   3. Noise Mix  (Excavator / Grinder / Drilling / ...)
         ↓
   4. Acoustic Augmentation (Gain / optional Clip)
         ↓
[Synthetic Noisy Construction Speech Dataset]
   clean/   OV2_000001_clean.wav
   noisy/   OV2_000001_n01.wav
             OV2_000001_n02.wav
   manifest.jsonl
```

**Target:** ~16,000 noisy samples (8,064 utterances × 2 versions)  
**Features:** Full Checkpoint & Instant Flush to Drive (100% Python 3.12 Compatible!)  
**Platform:** Google Colab (T4/CPU) or Kaggle

## Cell 1 — Mount Google Drive & Install Dependencies

In [ ]:
# ── Mount Drive (Colab only) ────────────────────────────────
import os
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/onevoice_audio_v1'
else:
    # Kaggle
    OUTPUT_ROOT = '/kaggle/working/onevoice_audio_v1'

print(f'Output root: {OUTPUT_ROOT}')

In [ ]:
# ── Install packages (Python 3.12 Compatible!) ──────────────
!pip install -q edge-tts soundfile librosa audiomentations pandas tqdm

## Cell 2 — Clone Dataset from GitHub

In [ ]:
# ── Clone OneVoice repo to get utterances CSV ───────────────
if not os.path.exists('/content/OneVoice'):
    !git clone --depth 1 https://github.com/Platypus27-coder/OneVoice.git /content/OneVoice
else:
    print('Repo already cloned.')

# Auto-detect data path (supports both root and subfolder structures)
if os.path.exists('/content/OneVoice/data/onevoice_construction_v2'):
    DATA_DIR = '/content/OneVoice/data/onevoice_construction_v2'
elif os.path.exists('/content/OneVoice/onevoice-edge/data/onevoice_construction_v2'):
    DATA_DIR = '/content/OneVoice/onevoice-edge/data/onevoice_construction_v2'
else:
    DATA_DIR = '/content/data/onevoice_construction_v2'

print('Data Path:', DATA_DIR)
print('Files:', os.listdir(DATA_DIR))

## Cell 3 — Configuration (Edit before running)

In [ ]:
import random, json
from pathlib import Path

# ── Output directories ─────────────────────────────────────
NOISE_DIR   = os.path.join(OUTPUT_ROOT, 'noise_bank')
CLEAN_DIR   = os.path.join(OUTPUT_ROOT, 'clean')
NOISY_DIR   = os.path.join(OUTPUT_ROOT, 'noisy')
MANIFEST    = os.path.join(OUTPUT_ROOT, 'manifest.jsonl')

for d in [OUTPUT_ROOT, NOISE_DIR, CLEAN_DIR, NOISY_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Sampling ───────────────────────────────────────────────
SAMPLES_PER_TEXT = 2          # each utterance → 2 noisy versions
SAMPLE_RATE      = 16000
MAX_UTTERANCES   = None       # None = all 8064; set int to limit for quick test

# ── Speaker pool (Microsoft Edge Neural Voices) ─────────────
VI_SPEAKERS = [
    ('vi-VN-HoaiMyNeural', None),  # Vietnamese Female
    ('vi-VN-NamMinhNeural', None), # Vietnamese Male
]
EN_SPEAKERS = [
    ('en-US-JennyNeural', None),   # US Female
    ('en-US-GuyNeural', None),     # US Male
    ('en-GB-SoniaNeural', None),   # UK Female
    ('en-AU-WilliamNeural', None), # AU Male
]

# ── Noise classes ──────────────────────────────────────────
NOISE_CLASSES = [
    'excavator.wav',
    'angle_grinder.wav',
    'drilling.wav',
    'hammer.wav',
    'diesel_engine.wav',
    'generator.wav',
    'truck.wav',
    'wind.wav',
    'worker_babble.wav',
]

# ── SNR levels ─────────────────────────────────────────────
SNR_OPTIONS = [0, 5, 10, 15, 20]

print('Config ready.')
print(f'  Output: {OUTPUT_ROOT}')
print(f'  Samples per utterance: {SAMPLES_PER_TEXT}')
print(f'  Max utterances: {MAX_UTTERANCES or "all"}')

## Cell 4 — Download Construction Noise Bank

> **Lưu ý:** Điền URL vào `NOISE_URLS` bên dưới.
> - **ESC-50** (CC BY 3.0): https://github.com/karolpiczak/ESC-50
> - **DEMAND** (CC BY-SA): https://zenodo.org/record/1227121
> - Hoặc upload thủ công lên `noise_bank/` rồi bỏ qua cell này.

In [ ]:
# ── Fill in real URLs or Google Drive file IDs ─────────────
# If you have files on Drive, use:
#   !cp /content/drive/MyDrive/noise/excavator.wav {NOISE_DIR}/excavator.wav

NOISE_URLS = {
    # 'excavator.wav':     'https://your-url-here',
    # 'angle_grinder.wav': 'https://your-url-here',
    # ... fill in the rest
}

def download_noise_bank():
    missing = []
    for fname, url in NOISE_URLS.items():
        dst = os.path.join(NOISE_DIR, fname)
        if not os.path.exists(dst):
            print(f'Downloading {fname}...')
            os.system(f'wget -q -O "{dst}" "{url}"')
        else:
            print(f'  {fname} already present.')
    # Check which noise files are actually present
    present = [f for f in NOISE_CLASSES if os.path.exists(os.path.join(NOISE_DIR, f))]
    missing = [f for f in NOISE_CLASSES if f not in present]
    print(f'\nPresent : {present}')
    if missing:
        print(f'Missing : {missing}  ← generation will skip these')
    return present

available_noises = download_noise_bank()

## Cell 5 — Core Audio Processing Functions

In [ ]:
import numpy as np
import soundfile as sf
import librosa
import asyncio
import edge_tts

# ── TTS synthesis using edge-tts ─────────────────────────────
def tts_synthesize(text: str, out_path: str,
                   voice_name: str, speaker: str = None) -> bool:
    try:
        async def _synth():
            communicate = edge_tts.Communicate(text, voice_name)
            await communicate.save(out_path)
        asyncio.run(_synth())
        return True
    except Exception as e:
        print(f'  [TTS ERR] {e}')
        return False

# ── Room Impulse Response ──────────────────────────────────
def apply_rir(speech: np.ndarray, sr: int,
              rir_path: str = None) -> np.ndarray:
    """Convolve with RIR or simulate simple reverb."""
    if rir_path and os.path.exists(rir_path):
        rir, _ = librosa.load(rir_path, sr=sr, mono=True)
        out = np.convolve(speech, rir, mode='full')[:len(speech)]
    else:
        # Lightweight simulated echo reverb
        delay = int(sr * random.uniform(0.03, 0.08))
        decay = random.uniform(0.2, 0.4)
        echo  = np.zeros_like(speech)
        echo[delay:] = speech[:-delay] * decay
        out = speech + echo
    peak = np.max(np.abs(out)) + 1e-9
    return out / peak

# ── Noise mixing at target SNR ─────────────────────────────
def mix_noise(speech: np.ndarray, noise: np.ndarray,
              snr_db: float) -> np.ndarray:
    # Loop noise if too short
    if len(noise) < len(speech):
        noise = np.tile(noise, int(np.ceil(len(speech) / len(noise))))
    start = random.randint(0, max(0, len(noise) - len(speech)))
    noise = noise[start: start + len(speech)]

    rms_s = np.sqrt(np.mean(speech ** 2) + 1e-9)
    rms_n = np.sqrt(np.mean(noise  ** 2) + 1e-9)
    scale = rms_s / (rms_n * (10 ** (snr_db / 20)))
    mixed = speech + scale * noise
    return np.clip(mixed / (np.max(np.abs(mixed)) + 1e-9), -1.0, 1.0)

# ── Gain & clip augmentation ───────────────────────────────
def augment(audio: np.ndarray,
            gain_range=(-3.0, 3.0), clip_prob=0.05) -> np.ndarray:
    gain = random.uniform(*gain_range)
    audio = audio * (10 ** (gain / 20))
    if random.random() < clip_prob:
        threshold = random.uniform(0.7, 0.95)
        audio = np.clip(audio, -threshold, threshold)
    return audio

print('Audio functions ready.')

## Cell 6 — Main Generation Loop (Checkpoint & Auto-Resume Enabled)

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

def generate_dataset():
    # ── Setup dirs ──────────────────────────────────────────
    for d in [CLEAN_DIR, NOISY_DIR, OUTPUT_ROOT]:
        os.makedirs(d, exist_ok=True)

    # ── Load existing manifest checkpoint (if resuming) ─────
    existing_audios = set()
    if os.path.exists(MANIFEST):
        with open(MANIFEST, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        data = json.loads(line)
                        existing_audios.add(data.get('audio'))
                    except Exception:
                        pass
        print(f'🔄 Checkpoint found: {len(existing_audios)} samples already processed.')

    # ── Load utterances ─────────────────────────────────────
    df = pd.read_csv(os.path.join(DATA_DIR, 'utterances_all.csv'))
    if MAX_UTTERANCES:
        df = df.head(MAX_UTTERANCES)
    print(f'Utterances loaded: {len(df)}')

    # ── Load noise bank into memory ─────────────────────────
    noise_cache = {}
    for nc in NOISE_CLASSES:
        path = os.path.join(NOISE_DIR, nc)
        if os.path.exists(path):
            audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
            noise_cache[nc] = audio
    if not noise_cache:
        print('[WARN] No noise files found in noise_bank. Using silence fallback.')
        noise_cache['silence.wav'] = np.zeros(SAMPLE_RATE)

    usable_noises = list(noise_cache.keys())
    sample_idx = len(existing_audios)
    skipped = 0

    # ── Open manifest in APPEND mode for instant streaming to Drive ─
    with open(MANIFEST, 'a', encoding='utf-8') as manifest_file:
        for _, row in tqdm(df.iterrows(), total=len(df), desc='Generating (Auto-Resume)'):
            utt_id   = str(row['utterance_id'])
            vi_text  = str(row['vi'])
            en_text  = str(row.get('en', ''))
            domain   = str(row.get('domain', 'unknown'))
            intent   = str(row.get('intent', 'unknown'))
            risk     = str(row.get('risk_level', 'unknown'))
            split    = str(row.get('split', 'train'))

            # Check if all variants for this utterance already exist on Drive
            all_done = True
            for v in range(SAMPLES_PER_TEXT):
                noisy_fname = f'{utt_id}_n{v+1:02d}.wav'
                noisy_path  = os.path.join(NOISY_DIR, noisy_fname)
                if noisy_fname not in existing_audios or not os.path.exists(noisy_path):
                    all_done = False
                    break
            if all_done:
                continue  # Fast skip fully processed utterance

            # ── Step 1: TTS → clean wav ─────────────────────────
            tts_model, speaker = random.choice(VI_SPEAKERS)
            clean_fname = f'{utt_id}_clean.wav'
            clean_path  = os.path.join(CLEAN_DIR, clean_fname)

            if not os.path.exists(clean_path):
                ok = tts_synthesize(vi_text, clean_path, tts_model, speaker)
                if not ok:
                    skipped += 1
                    continue

            try:
                clean_audio, _ = librosa.load(clean_path, sr=SAMPLE_RATE, mono=True)
            except Exception:
                skipped += 1
                continue

            # ── Step 2: RIR / Reverb ────────────────────────────
            reverbed = apply_rir(clean_audio, SAMPLE_RATE)

            # ── Step 3: SAMPLES_PER_TEXT noisy variants ─────────
            for v in range(SAMPLES_PER_TEXT):
                noisy_fname = f'{utt_id}_n{v+1:02d}.wav'
                noisy_path  = os.path.join(NOISY_DIR, noisy_fname)

                if noisy_fname in existing_audios and os.path.exists(noisy_path):
                    continue

                noise_name  = random.choice(usable_noises)
                snr_db      = random.choice(SNR_OPTIONS)
                use_reverb  = random.random() > 0.35

                base = reverbed if use_reverb else clean_audio
                mixed = mix_noise(base, noise_cache[noise_name], snr_db)
                mixed = augment(mixed)

                sf.write(noisy_path, mixed, SAMPLE_RATE)

                entry = {
                    'audio':               noisy_fname,
                    'clean_audio':         clean_fname,
                    'text':                vi_text,
                    'translation':         en_text,
                    'domain':              domain,
                    'intent':              intent,
                    'risk_level':          risk,
                    'split':               split,
                    'speaker_id':          f'{tts_model}__{speaker or "default"}',
                    'noise_type':          noise_name.replace('.wav', ''),
                    'snr_db':              snr_db,
                    'reverb':              use_reverb,
                    'rir_id':              'simulated_echo' if use_reverb else 'none',
                    'synthetic_speech':    True,
                    'synthetic_noise_mix': True,
                    'sample_rate':         SAMPLE_RATE,
                }
                # Instant append + flush to Drive!
                manifest_file.write(json.dumps(entry, ensure_ascii=False) + '\n')
                manifest_file.flush()

                existing_audios.add(noisy_fname)
                sample_idx += 1

    print(f'\n✅ Done!')
    print(f'   Total samples in manifest : {sample_idx}')
    print(f'   Skipped (TTS error)       : {skipped}')
    print(f'   Clean dir  : {CLEAN_DIR}')
    print(f'   Noisy dir  : {NOISY_DIR}')
    print(f'   Manifest   : {MANIFEST}')

generate_dataset()

## Cell 7 — Verify Stats

In [ ]:
import pandas as pd

entries = []
with open(MANIFEST, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            entries.append(json.loads(line))

df_m = pd.DataFrame(entries)
print(f'=== Synthetic Noisy Construction Speech Dataset ===')
print(f'Total samples     : {len(df_m)}')
print(f'\nBy domain:')
print(df_m['domain'].value_counts().to_string())
print(f'\nBy intent:')
print(df_m['intent'].value_counts().to_string())
print(f'\nBy noise_type:')
print(df_m['noise_type'].value_counts().to_string())
print(f'\nBy snr_db:')
print(df_m['snr_db'].value_counts().sort_index().to_string())
print(f'\nBy split:')
print(df_m['split'].value_counts().to_string())

## Cell 8 — Quick Listen Test (Colab only)

In [ ]:
# Play a random clean vs noisy pair to verify quality
from IPython.display import Audio, display
import random

sample = random.choice(entries)
print(f"Text  : {sample['text']}")
print(f"Noise : {sample['noise_type']}  |  SNR: {sample['snr_db']} dB  |  Reverb: {sample['reverb']}")
print(f"Domain: {sample['domain']}  |  Intent: {sample['intent']}")
print('\n--- Clean ---')
display(Audio(os.path.join(CLEAN_DIR, sample['clean_audio']), rate=SAMPLE_RATE))
print('--- Noisy ---')
display(Audio(os.path.join(NOISY_DIR, sample['audio']), rate=SAMPLE_RATE))